# 4. Diffusion Models - Code Implementations

This notebook provides minimal, educational PyTorch implementations corresponding to the mathematical equations presented in the **4. Diffusion Models** lecture slides.

The focus here is **not** on a complete training pipeline, data loading, or model architectures (like U-Net). Instead, we focus on translating the core mathematical equations directly into elegant Python code, step-by-step.


In [ ]:
import torch

# ---------------------------------------------------------
# 1. Dummy Neural Networks
# ---------------------------------------------------------
# We mock the neural networks so the mathematical functions can run without errors.
# In practice, these would be deep neural networks (e.g., U-Net).

def epsilon_theta(x_t, t):
    """Predicts the injected noise at step t. (DDPM, DDIM)"""
    return torch.randn_like(x_t)

def s_theta(x_t, t):
    """Predicts the score (gradient of log density) at step t. (Score Matching)"""
    return torch.randn_like(x_t)


# ---------------------------------------------------------
# 2. Diffusion Schedule Setup
# ---------------------------------------------------------
T = 1000                                      # Total timesteps
betas = torch.linspace(1e-4, 0.02, T)         # Variance schedule (beta_t)
alphas = 1.0 - betas                          # alpha_t = 1 - beta_t
alphas_bar = torch.cumprod(alphas, dim=0)     # alpha_bar_t = product of alphas

print("Environment and dummy networks initialized.")

## Forward Process (Slide 8)
The forward process adds noise to the data. We can either do this step-by-step (Markov Chain) or jump to an arbitrary step directly.

**1. Markov Chain step:**
$$x_t = \sqrt{1-\beta_t}\,x_{t-1} + \sqrt{\beta_t}\, \epsilon$$

**2. Arbitrary step $t$ directly from $x_0$:**
$$q(x_t \mid x_0) = \mathcal{N}\!\left( x_t;\, \sqrt{\bar{\alpha}_t}\,x_0,\, (1-\bar{\alpha}_t) I \right)$$
$$x_t = \sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\epsilon$$


In [ ]:
def forward_step_markov(x_t_minus_1, t):
    """
    Corrupts data by a single discrete step.
    Equation: x_t = sqrt(1 - beta_t) * x_{t-1} + sqrt(beta_t) * epsilon
    """
    beta_t = betas[t]
    epsilon = torch.randn_like(x_t_minus_1) # epsilon ~ N(0, I)

    x_t = torch.sqrt(1 - beta_t) * x_t_minus_1 + torch.sqrt(beta_t) * epsilon
    return x_t

def forward_step_arbitrary(x_0, t):
    """
    Jumps directly from clean data x_0 to noisy data x_t.
    Equation: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon
    """
    alpha_bar_t = alphas_bar[t]
    epsilon = torch.randn_like(x_0) # epsilon ~ N(0, I)

    x_t = torch.sqrt(alpha_bar_t) * x_0 + torch.sqrt(1 - alpha_bar_t) * epsilon
    return x_t, epsilon

## Reverse Process (Slide 10)
At generation, the injected noise $\epsilon$ is unknown. We train a network $\epsilon_\theta(x_t, t)$ to predict it, parameterising the mean as:

$$\mu_\theta(x_t, t) = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right)$$


In [ ]:
def compute_mu_theta(x_t, t):
    """
    Computes the predicted mean for the reverse step.
    Equation: mu_theta = 1 / sqrt(alpha_t) * [ x_t - (1 - alpha_t) / sqrt(1 - alpha_bar_t) * epsilon_theta ]
    """
    alpha_t = alphas[t]
    alpha_bar_t = alphas_bar[t]

    # Predict the noise using our neural network
    pred_epsilon = epsilon_theta(x_t, t)

    mu = (1 / torch.sqrt(alpha_t)) * (
        x_t - ((1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)) * pred_epsilon
    )
    return mu

## DDPM: Training (Slide 11)
**Training Objective:** Predict the added noise.
$$\mathcal{L} = \mathbb{E}_{t, x_0, \epsilon} \left[ \|\epsilon - \epsilon_\theta(x_t, t)\|_2^2 \right]$$


In [ ]:
def ddpm_training_loss(x_0):
    """
    Computes the MSE loss for training the noise prediction network.
    Algorithm steps:
    1. Sample t ~ U({1, ..., T}), and epsilon ~ N(0, I).
    2. Compute noisy data: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon
    3. Compute loss: || epsilon - epsilon_theta(x_t, t) ||^2
    """
    # 1. Sample a random timestep and noise
    t = torch.randint(0, T, (1,)).item()
    epsilon = torch.randn_like(x_0)

    # 2. Add noise to the data (Forward process)
    alpha_bar_t = alphas_bar[t]
    x_t = torch.sqrt(alpha_bar_t) * x_0 + torch.sqrt(1 - alpha_bar_t) * epsilon

    # 3. Predict the noise and calculate the Mean Squared Error (MSE)
    pred_epsilon = epsilon_theta(x_t, t)
    loss = torch.nn.functional.mse_loss(pred_epsilon, epsilon)

    return loss

## DDPM: Sampling (Slide 12)
To generate data, we start with pure noise and iteratively denoise it.

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right) + \sigma_t z$$
*(where $z \sim \mathcal{N}(0, I)$ and $\sigma_t^2 = \beta_t$)*


In [ ]:
def ddpm_sampling_step(x_t, t):
    """
    Performs a single reverse denoising step in DDPM.
    """
    alpha_t = alphas[t]
    alpha_bar_t = alphas_bar[t]
    sigma_t = torch.sqrt(betas[t]) # Using sigma_t^2 = beta_t

    # Compute the mean mu_theta
    pred_epsilon = epsilon_theta(x_t, t)
    mu = (1 / torch.sqrt(alpha_t)) * (
        x_t - ((1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)) * pred_epsilon
    )

    # Add variance/noise (z = 0 if t = 0)
    z = torch.randn_like(x_t) if t > 0 else torch.zeros_like(x_t)

    x_t_minus_1 = mu + sigma_t * z
    return x_t_minus_1

## DDIM: Generalised Sampling (Slide 14)
DDIM uses a non-Markovian forward process. We can jump back by multiple steps, and control the stochasticity via $\sigma_t$.

$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \underbrace{\Bigg( \frac{x_t - \sqrt{1-\bar{\alpha}_t}\,\epsilon_\theta(x_t, t)}{\sqrt{\bar{\alpha}_t}} \Bigg)}_{\text{Predicted } x_0} + \underbrace{\sqrt{1-\bar{\alpha}_{t-1} - \sigma_t^2}\, \epsilon_\theta(x_t, t)}_{\text{Direction to } x_t} + \underbrace{\sigma_t z}_{\text{Noise}}$$


In [ ]:
def ddim_sampling_step(x_t, t, t_prev, sigma_t):
    """
    Performs a generalised sampling step. By varying sigma_t, we can interpolate
    between DDPM (stochastic) and DDIM (deterministic).
    """
    alpha_bar_t = alphas_bar[t]
    # Handle the edge case when t_prev < 0 (i.e. generating the final x_0)
    alpha_bar_t_prev = alphas_bar[t_prev] if t_prev >= 0 else torch.tensor(1.0)

    pred_epsilon = epsilon_theta(x_t, t)

    # Term 1: Predicted x_0
    pred_x0 = (x_t - torch.sqrt(1 - alpha_bar_t) * pred_epsilon) / torch.sqrt(alpha_bar_t)

    # Term 2: Direction pointing to x_t
    direction = torch.sqrt(1 - alpha_bar_t_prev - sigma_t**2) * pred_epsilon

    # Term 3: Random Noise
    z = torch.randn_like(x_t) if t_prev >= 0 and sigma_t > 0 else torch.zeros_like(x_t)
    noise = sigma_t * z

    # Combine all terms
    x_prev = torch.sqrt(alpha_bar_t_prev) * pred_x0 + direction + noise
    return x_prev

## DDPM vs. DDIM (Slide 15)
Setting $\sigma_t = 0$ removes the noise term, yielding the deterministic DDIM process:

$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}}\,\hat{x}_0 + \sqrt{1-\bar{\alpha}_{t-1}}\,\epsilon_\theta(x_t, t)$$


In [ ]:
def ddim_deterministic_step(x_t, t, t_prev):
    """
    The completely deterministic DDIM step (sigma_t = 0).
    Allows mapping latents to data and back (invertible).
    """
    alpha_bar_t = alphas_bar[t]
    alpha_bar_t_prev = alphas_bar[t_prev] if t_prev >= 0 else torch.tensor(1.0)

    pred_epsilon = epsilon_theta(x_t, t)

    # hat{x}_0 (Predicted x_0)
    hat_x0 = (x_t - torch.sqrt(1 - alpha_bar_t) * pred_epsilon) / torch.sqrt(alpha_bar_t)

    # Deterministic mapping (no random noise term)
    x_prev = torch.sqrt(alpha_bar_t_prev) * hat_x0 + torch.sqrt(1 - alpha_bar_t_prev) * pred_epsilon
    return x_prev

## Score Matching: SDE & PF-ODE (Slide 16 & 17)
When we take the number of steps to infinity, we get continuous-time Stochastic Differential Equations (SDEs).

**SDE (Euler-Maruyama step):**
$$x_{t-\Delta t} = x_t - \Delta t \big[ f - g^2 s(x_t, t) \big] + g\sqrt{\Delta t}\, z$$

**Probability Flow ODE (Euler step):**
$$x_{t-\Delta t} = x_t - \Delta t \big[ f - \frac{1}{2}g^2 s(x_t, t) \big]$$


In [ ]:
def f_drift(x, t):
    """Example drift coefficient f(x, t)"""
    return -0.5 * x

def g_diffusion(t):
    """Example diffusion coefficient g(t)"""
    return torch.tensor(0.1)

def reverse_sde_step(x_t, t, delta_t):
    """
    A discrete reverse generation step using the Euler-Maruyama solver for SDEs.
    """
    f = f_drift(x_t, t)
    g = g_diffusion(t)
    s = s_theta(x_t, t)   # Our score prediction network
    z = torch.randn_like(x_t)

    drift_term = f - (g**2) * s
    noise_term = g * torch.sqrt(torch.tensor(delta_t)) * z

    x_prev = x_t - delta_t * drift_term + noise_term
    return x_prev

def pf_ode_step(x_t, t, delta_t):
    """
    A discrete reverse generation step using the Euler solver for the Probability Flow ODE.
    """
    f = f_drift(x_t, t)
    g = g_diffusion(t)
    s = s_theta(x_t, t)

    drift_term = f - 0.5 * (g**2) * s

    x_prev = x_t - delta_t * drift_term
    return x_prev

## Denoising Score Matching (Slide 18 & 19)
Predicting noise is mathematically equivalent to estimating the score.

**Relationship:**
$$\epsilon_\theta(x_t, t) = -\sqrt{1-\bar{\alpha}_t}\; s_\theta(x_t, t)$$

**Training (Score Estimation):**
$$\left\| s_\theta(x_t, t) - \left( -\frac{\epsilon}{\sqrt{1-\bar{\alpha}_t}} \right) \right\|^2$$


In [ ]:
def compute_score_from_noise(x_t, t):
    """
    Extracts the score prediction from a standard DDPM noise-prediction network.
    Equation: s_theta(x_t, t) = - epsilon_theta(x_t, t) / sqrt(1 - alpha_bar_t)
    """
    alpha_bar_t = alphas_bar[t]
    score = -epsilon_theta(x_t, t) / torch.sqrt(1 - alpha_bar_t)
    return score

def dsm_training_loss(x_0):
    """
    Training objective to directly match the score, rather than predicting noise.
    """
    t = torch.randint(0, T, (1,)).item()
    epsilon = torch.randn_like(x_0)

    # Corrupt data
    alpha_bar_t = alphas_bar[t]
    x_t = torch.sqrt(alpha_bar_t) * x_0 + torch.sqrt(1 - alpha_bar_t) * epsilon

    # Train s_theta to match the exact score
    true_score = -epsilon / torch.sqrt(1 - alpha_bar_t)
    pred_score = s_theta(x_t, t)

    loss = torch.nn.functional.mse_loss(pred_score, true_score)
    return loss

In [ ]:
# ---------------------------------------------------------
# Denoising Score Matching: Training and Inference Demo
# Corresponds to Slide 20
# ---------------------------------------------------------

def demonstrate_denoising_score_matching(x_0, delta_t=1e-3):
    """
    Illustrates the complete workflow from Slide 20:

    Training:
        1. Sample t and epsilon.
        2. Corrupt x_0 to obtain x_t.
        3. Compute the exact conditional score target.
        4. Compare it with the network prediction.

    Inference:
        5. Use the predicted score in one reverse-SDE step.
        6. Use the predicted score in one PF-ODE step.
    """

    # =====================================================
    # Part 1: Training / score estimation
    # =====================================================

    # Sample a diffusion timestep and Gaussian noise
    t = torch.randint(0, T, (1,)).item()
    epsilon = torch.randn_like(x_0)

    alpha_bar_t = alphas_bar[t].to(
        device=x_0.device,
        dtype=x_0.dtype
    )

    # Corrupt the clean data:
    # x_t = sqrt(alpha_bar_t) * x_0
    #       + sqrt(1 - alpha_bar_t) * epsilon
    x_t = (
        torch.sqrt(alpha_bar_t) * x_0
        + torch.sqrt(1.0 - alpha_bar_t) * epsilon
    )

    # Exact conditional score:
    # grad_{x_t} log p(x_t | x_0)
    true_score = -epsilon / torch.sqrt(1.0 - alpha_bar_t)

    # Score predicted by the neural network
    predicted_score = s_theta(x_t, t)

    # Denoising score-matching loss
    score_loss = torch.nn.functional.mse_loss(
        predicted_score,
        true_score
    )

    # =====================================================
    # Part 2: Inference / generation
    # =====================================================

    # One stochastic reverse-SDE step
    x_previous_sde = reverse_sde_step(
        x_t=x_t,
        t=t,
        delta_t=delta_t
    )

    # One deterministic Probability Flow ODE step
    x_previous_ode = pf_ode_step(
        x_t=x_t,
        t=t,
        delta_t=delta_t
    )

    print(f"Sampled timestep: t = {t}")
    print(f"Score-matching loss: {score_loss.item():.6f}")
    print()
    print("Input shapes:")
    print(f"  x_0:                 {tuple(x_0.shape)}")
    print(f"  x_t:                 {tuple(x_t.shape)}")
    print(f"  true score:          {tuple(true_score.shape)}")
    print()
    print("Reverse-step outputs:")
    print(f"  Reverse SDE output:  {tuple(x_previous_sde.shape)}")
    print(f"  PF-ODE output:       {tuple(x_previous_ode.shape)}")

    return {
        "t": t,
        "x_0": x_0,
        "x_t": x_t,
        "epsilon": epsilon,
        "true_score": true_score,
        "predicted_score": predicted_score,
        "loss": score_loss,
        "x_previous_sde": x_previous_sde,
        "x_previous_ode": x_previous_ode,
    }


# Example input: a small batch of image-like tensors
x_0_example = torch.randn(4, 3, 32, 32)

results = demonstrate_denoising_score_matching(
    x_0_example,
    delta_t=1e-3
)